In [7]:
%load_ext autoreload
%autoreload 2
import time
import datetime

import torch
from ogb.linkproppred import PygLinkPropPredDataset, Evaluator

# NCN predictor
from src.model import CNLinkPredictor, GCN
from src.ogbdataset import loaddataset
from src.train_test_func import train, test

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# cora
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is', device)
print('today is', datetime.date.today())
use_valedges_as_input = False
data, split_edge = loaddataset('Cora', use_valedges_as_input=use_valedges_as_input, load=None)
data = data.to(device)

# model(GCN)
hid_dim = 256
mplayers = 1     # number of message passing layers
model = GCN(data.num_features, hidden_channels=hid_dim, out_channels=hid_dim, num_layers=mplayers,
            dropout=0.3, xdropout=0.7, taildropout=0.3, jk=True).to(device)

# Neural common neighbors(NCN)
predictor = CNLinkPredictor(in_channels=hid_dim, hidden_channels=hid_dim, out_channels=1, dropout=0.3, edrop=0.3, ln=True, use_xlin=True, tailact=True).to(device)
optimizer = torch.optim.Adam([{'params': model.parameters(), "lr": 0.0043}, {'params': predictor.parameters(), 'lr': 0.0024}])

# evaluator
evaluator = Evaluator(name=f'ogbl-ppa')

# train
start_time = time.time()
epochs = 100
batch_size = 1152
maskinput = True # whether to use target link removal
increasealpha = False
for epoch in range(1, 1 + epochs):
    alpha = max(0, min((epoch-5)*0.1, 1)) if increasealpha else None
    loss = train(model, predictor, data, split_edge, optimizer,
                batch_size, maskinput, [], alpha)
    
    if epoch % 20 == 0:
        print(f"Epoch: {epoch}, loss: {loss:.4f}")
    
    test_batch_size = 1152
    results, h = test(model, predictor, data, split_edge, evaluator,
                               test_batch_size, use_valedges_as_input)

print(f"final test AUC: {results['auc'][-1]: .4f}, test_AP: {results['ap'][-1]: .4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time:.4f}")


device is cuda
today is 2026-07-19
2708 tensor(2707)
dataset split 
train edge 3696
valid edge 527
valid edge_neg 1055
test edge 1055
test edge_neg 1055
Epoch: 20, loss: 1.0326
Epoch: 40, loss: 0.8347
Epoch: 60, loss: 0.7368
Epoch: 80, loss: 0.6969
Epoch: 100, loss: 0.6600
final test AUC:  0.9259, test_AP:  0.9351
--------총 걸린시간(s): 14.2538


In [ ]:
# Citeseer
print('data is Citeseer')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is ', device)
print('today is', datetime.date.today())
use_valedges_as_input = False
data, split_edge = loaddataset('Citeseer', use_valedges_as_input=use_valedges_as_input, load=None)
data = data.to(device)

# model(GCN)
hid_dim = 256
mplayers = 1     # number of message passing layers
model = GCN(data.num_features, hidden_channels=hid_dim, out_channels=hid_dim, num_layers=mplayers,
            dropout=0.3, xdropout=0.7, taildropout=0.3, jk=True).to(device)

# Neural common neighbors(NCN)
predictor = CNLinkPredictor(in_channels=hid_dim, hidden_channels=hid_dim, out_channels=1, dropout=0.3, edrop=0.3, ln=True, use_xlin=True, tailact=True).to(device)
optimizer = torch.optim.Adam([{'params': model.parameters(), "lr": 0.0043}, {'params': predictor.parameters(), 'lr': 0.0024}])

# evaluator
evaluator = Evaluator(name=f'ogbl-ppa')

# train
start_time = time.time()
epochs = 100
batch_size = 1152
maskinput = True # whether to use target link removal
increasealpha = False
for epoch in range(1, 1 + epochs):
    alpha = max(0, min((epoch-5)*0.1, 1)) if increasealpha else None
    loss = train(model, predictor, data, split_edge, optimizer,
                batch_size, maskinput, [], alpha)
    
    if epoch % 20 == 0:
        print(f"Epoch: {epoch}, loss: {loss:.4f}")
    
    test_batch_size = 1152
    results, h = test(model, predictor, data, split_edge, evaluator,
                               test_batch_size, use_valedges_as_input)

print(f"final test AUC: {results['auc'][-1]: .4f}, test_AP: {results['ap'][-1]: .4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time:.4f}")



data is Citeseer
device is  cuda
today is 2026-07-19


c:\Users\pmw\AppData\Local\pypoetry\Cache\virtualenvs\ncn-7KNnx3LI-py3.11\Lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


3327 tensor(3326)
dataset split 
train edge 3187
valid edge 455
valid edge_neg 910
test edge 910
test edge_neg 910
Epoch: 20, loss: 1.0568
Epoch: 40, loss: 0.7510
Epoch: 60, loss: 0.6025
Epoch: 80, loss: 0.5646
Epoch: 100, loss: 0.5101
final test AUC:  0.8873, test_AP:  0.9120
--------총 걸린시간(s): 11.4481


In [10]:
# Pubmed
print('data is Pubmed')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device is ', device)
print('today is', datetime.date.today())
use_valedges_as_input = False
data, split_edge = loaddataset('Pubmed', use_valedges_as_input=use_valedges_as_input, load=None)
data = data.to(device)

# model(GCN)
hid_dim = 256
mplayers = 1     # number of message passing layers
model = GCN(data.num_features, hidden_channels=hid_dim, out_channels=hid_dim, num_layers=mplayers,
            dropout=0.3, xdropout=0.7, taildropout=0.3, jk=True).to(device)

# Neural common neighbors(NCN)
predictor = CNLinkPredictor(in_channels=hid_dim, hidden_channels=hid_dim, out_channels=1, dropout=0.3, edrop=0.3, ln=True, use_xlin=True, tailact=True).to(device)
optimizer = torch.optim.Adam([{'params': model.parameters(), "lr": 0.0043}, {'params': predictor.parameters(), 'lr': 0.0024}])

# evaluator
evaluator = Evaluator(name=f'ogbl-ppa')

# train
start_time = time.time()
epochs = 100
batch_size = 1152
maskinput = True # whether to use target link removal
increasealpha = False
for epoch in range(1, 1 + epochs):
    alpha = max(0, min((epoch-5)*0.1, 1)) if increasealpha else None
    loss = train(model, predictor, data, split_edge, optimizer,
                batch_size, maskinput, [], alpha)
    
    if epoch % 20 == 0:
        print(f"Epoch: {epoch}, loss: {loss:.4f}")
    
    test_batch_size = 1152
    results, h = test(model, predictor, data, split_edge, evaluator,
                               test_batch_size, use_valedges_as_input)

print(f"final test AUC: {results['auc'][-1]: .4f}, test_AP: {results['ap'][-1]: .4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time:.4f}")


data is Pubmed
device is  cuda
today is 2026-07-19


c:\Users\pmw\AppData\Local\pypoetry\Cache\virtualenvs\ncn-7KNnx3LI-py3.11\Lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


19717 tensor(19715)
dataset split 
train edge 31028
valid edge 4432
valid edge_neg 8864
test edge 8864
test edge_neg 8864
Epoch: 20, loss: 0.5759
Epoch: 40, loss: 0.5450
Epoch: 60, loss: 0.5235
Epoch: 80, loss: 0.5095
Epoch: 100, loss: 0.5046
final test AUC:  0.9807, test_AP:  0.9786
--------총 걸린시간(s): 125.8518
